In [10]:
import os
import pandas as pd

In [11]:
# RAVDESS encodes emotion as the 3rd number in the filename
# Format: modality-channel-emotion-intensity-statement-repetition-actor.wav
RAVDESS_EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

# SAVEE filenames start with a letter code for emotion
# Format: DC_a01.wav (speaker_emotionNumber.wav)
SAVEE_EMOTION_MAP = {
    "a": "angry",
    "d": "disgust",
    "f": "fearful",
    "h": "happy",
    "n": "neutral",
    "sa": "sad",
    "su": "surprised",
}

TESS_EMOTION_MAP = {
    "angry": "angry",
    "disgust": "disgust",
    "fear": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "ps": "surprised",
    "surprise": "surprised",           
    "pleasant_surprise": "surprised",  
    "sad": "sad",
}

**Dataset Specific Loaders**

In [12]:
def load_ravdess(data_dir):
    """
    RAVDESS structure:
        data_dir/
            Actor_01/
                03-01-01-01-01-01-01.wav
                03-01-02-01-01-01-01.wav
                ...
            Actor_02/
                ...

    Filename format: modality-channel-emotion-intensity-statement-repetition-actor.wav
    - modality: 01=full-AV, 02=video-only, 03=audio-only
    - emotion: 01-08 (see RAVDESS_EMOTION_MAP)
    - actor: odd=male, even=female
    """
    records = []

    if not os.path.exists(data_dir):
        print(f"WARNING: RAVDESS directory not found at {data_dir}")
        return records

    for actor_folder in sorted(os.listdir(data_dir)):
        actor_path = os.path.join(data_dir, actor_folder)
        if not os.path.isdir(actor_path):
            continue

        for filename in sorted(os.listdir(actor_path)):
            if not filename.endswith(".wav"):
                continue

            file_path = os.path.join(actor_path, filename)
            parts = filename.replace(".wav", "").split("-")

            if len(parts) != 7:
                print(f"  Skipping unexpected filename format: {filename}")
                continue

            emotion_code = parts[2]
            actor_number = int(parts[6])

            emotion = RAVDESS_EMOTION_MAP.get(emotion_code, "unknown")
            gender = "female" if actor_number % 2 == 0 else "male"
            speaker_id = f"ravdess_{actor_number:02d}"

            records.append({
                "file_path": file_path,
                "emotion": emotion,
                "speaker_id": speaker_id,
                "gender": gender,
                "dataset": "ravdess",
            })

    print(f"RAVDESS: Loaded {len(records)} clips")
    return records

In [13]:
def load_tess(data_dir):
    """
    TESS structure (common Kaggle layout):
        data_dir/
            OAF_angry/
                OAF_back_angry.wav
                ...
            OAF_disgust/
                ...
            YAF_angry/
                ...

    OAF = Older Adult Female, YAF = Young Adult Female
    The emotion is the last word in the folder name.

    Alternative layout (flat):
        data_dir/
            OAF_back_angry.wav
            YAF_bar_happy.wav
            ...
    """
    records = []

    if not os.path.exists(data_dir):
        print(f"WARNING: TESS directory not found at {data_dir}")
        return records

    # Determine if the layout is folder-based or flat
    subdirs = [d for d in os.listdir(data_dir)
               if os.path.isdir(os.path.join(data_dir, d))]

    if subdirs:
        # Folder-based layout
        for folder_name in sorted(subdirs):
            folder_path = os.path.join(data_dir, folder_name)

            # Extract emotion from folder name (last part after underscore)
            folder_lower = folder_name.lower()
            emotion_word = folder_lower.split("_")[-1] if "_" in folder_lower else folder_lower

            emotion = TESS_EMOTION_MAP.get(emotion_word, "unknown")
            if emotion == "unknown":
                # Try matching partial strings
                for key, val in TESS_EMOTION_MAP.items():
                    if key in folder_lower:
                        emotion = val
                        break

            # Determine speaker from prefix
            if "oaf" in folder_lower:
                speaker_id = "tess_OAF"
            elif "yaf" in folder_lower:
                speaker_id = "tess_YAF"
            else:
                speaker_id = "tess_unknown"

            for filename in sorted(os.listdir(folder_path)):
                if not filename.lower().endswith(".wav"):
                    continue

                file_path = os.path.join(folder_path, filename)
                records.append({
                    "file_path": file_path,
                    "emotion": emotion,
                    "speaker_id": speaker_id,
                    "gender": "female",  # TESS only has female speakers
                    "dataset": "tess",
                })
    else:
        # Flat layout — emotion is the last word in the filename
        for filename in sorted(os.listdir(data_dir)):
            if not filename.lower().endswith(".wav"):
                continue

            file_path = os.path.join(data_dir, filename)
            name_lower = filename.lower().replace(".wav", "")
            parts = name_lower.split("_")

            # Last part is the emotion
            emotion_word = parts[-1] if parts else "unknown"
            emotion = TESS_EMOTION_MAP.get(emotion_word, "unknown")

            # Speaker from prefix
            if parts[0] == "oaf":
                speaker_id = "tess_OAF"
            elif parts[0] == "yaf":
                speaker_id = "tess_YAF"
            else:
                speaker_id = "tess_unknown"

            records.append({
                "file_path": file_path,
                "emotion": emotion,
                "speaker_id": speaker_id,
                "gender": "female",
                "dataset": "tess",
            })

    print(f"TESS: Loaded {len(records)} clips")
    return records

In [14]:
def load_savee(data_dir):
    """
    SAVEE structure:
        data_dir/
            DC_a01.wav     (speaker DC, angry, clip 01)
            DC_d01.wav     (speaker DC, disgust, clip 01)
            DC_f01.wav     (speaker DC, fearful, clip 01)
            DC_h01.wav     (speaker DC, happy, clip 01)
            DC_n01.wav     (speaker DC, neutral, clip 01)
            DC_sa01.wav    (speaker DC, sad, clip 01)
            DC_su01.wav    (speaker DC, surprised, clip 01)
            JE_a01.wav
            ...

    Speakers: DC, JE, JK, KL (all male)
    Emotion code: first letter(s) after the underscore, before the digits
    """
    records = []

    if not os.path.exists(data_dir):
        print(f"WARNING: SAVEE directory not found at {data_dir}")
        return records

    for filename in sorted(os.listdir(data_dir)):
        if not filename.lower().endswith(".wav"):
            continue

        file_path = os.path.join(data_dir, filename)
        name = filename.replace(".wav", "")

        # Split on underscore: speaker_emotionCode
        if "_" not in name:
            print(f"  Skipping unexpected filename format: {filename}")
            continue

        parts = name.split("_")
        speaker_code = parts[0]
        emotion_part = parts[1] if len(parts) > 1 else ""

        # Extract emotion letters (everything before the digits)
        emotion_code = ""
        for char in emotion_part:
            if char.isdigit():
                break
            emotion_code += char

        emotion = SAVEE_EMOTION_MAP.get(emotion_code, "unknown")
        speaker_id = f"savee_{speaker_code}"

        records.append({
            "file_path": file_path,
            "emotion": emotion,
            "speaker_id": speaker_id,
            "gender": "male",  # SAVEE only has male speakers
            "dataset": "savee",
        })

    print(f"SAVEE: Loaded {len(records)} clips")
    return records

**Main Loader: Combine All Datasets**

In [15]:
def load_all_datasets(ravdess_dir="data/ravdess",
                      tess_dir="data/tess",
                      savee_dir="data/savee"):
    """
    Load all three datasets and return a unified DataFrame.

    Parameters
    ----------
    ravdess_dir : str — path to RAVDESS root folder (contains Actor_01, Actor_02, ...)
    tess_dir    : str — path to TESS root folder
    savee_dir   : str — path to SAVEE root folder (contains .wav files directly)

    Returns
    -------
    pd.DataFrame with columns: file_path, emotion, speaker_id, gender, dataset
    """
    print("=" * 50)
    print("Loading datasets...")
    print("=" * 50)

    all_records = []
    all_records.extend(load_ravdess(ravdess_dir))
    all_records.extend(load_tess(tess_dir))
    all_records.extend(load_savee(savee_dir))

    df = pd.DataFrame(all_records)

    # Validate: check for unknown emotions
    unknowns = df[df["emotion"] == "unknown"]
    if len(unknowns) > 0:
        print(f"\nWARNING: {len(unknowns)} clips have 'unknown' emotion labels!")
        print(unknowns["file_path"].head(10).tolist())

    # Summary
    print(f"\n{'=' * 50}")
    print(f"Total clips loaded: {len(df)}")
    print(f"Unique emotions:    {sorted(df['emotion'].unique())}")
    print(f"Unique speakers:    {df['speaker_id'].nunique()}")
    print(f"Gender breakdown:   {df['gender'].value_counts().to_dict()}")
    print(f"Dataset breakdown:  {df['dataset'].value_counts().to_dict()}")
    print(f"{'=' * 50}")

    return df

**Emotion Filtering**

RAVDESS has 8 emotions including 'calm', which TESS and SAVEE don't have. So we'll keep only the 7 emotions shared across all three datasets: angry, disgust, fearful, happy, neutral, sad, surprised. We'll do this to keep a clean common label set. 

In [16]:
def filter_common_emotions(df):
    
    common_emotions = ["angry", "disgust", "fearful", "happy", "neutral", "sad", "surprised"]
    df_filtered = df[df["emotion"].isin(common_emotions)].copy()

    removed = len(df) - len(df_filtered)
    if removed > 0:
        print(f"Filtered to common emotions. Removed {removed} clips (likely 'calm' from RAVDESS).")
    print(f"Remaining: {len(df_filtered)} clips across {sorted(df_filtered['emotion'].unique())}")

    return df_filtered

In [17]:
df = load_all_datasets(
        ravdess_dir="data/ravdess",
        tess_dir="data/tess",
        savee_dir="data/savee",
)

# Optional: filter to 7 common emotions
df = filter_common_emotions(df)

# Save master DataFrame
df.to_csv("master_dataframe.csv", index=False)
print(f"\nSaved master DataFrame to master_dataframe.csv")
print(df.head(10))

Loading datasets...
RAVDESS: Loaded 1440 clips
TESS: Loaded 2800 clips
SAVEE: Loaded 480 clips

Total clips loaded: 4720
Unique emotions:    ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Unique speakers:    30
Gender breakdown:   {'female': 3520, 'male': 1200}
Dataset breakdown:  {'tess': 2800, 'ravdess': 1440, 'savee': 480}
Filtered to common emotions. Removed 192 clips (likely 'calm' from RAVDESS).
Remaining: 4528 clips across ['angry', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']

Saved master DataFrame to master_dataframe.csv
                                         file_path  emotion  speaker_id  \
0   data/ravdess\Actor_01\03-01-01-01-01-01-01.wav  neutral  ravdess_01   
1   data/ravdess\Actor_01\03-01-01-01-01-02-01.wav  neutral  ravdess_01   
2   data/ravdess\Actor_01\03-01-01-01-02-01-01.wav  neutral  ravdess_01   
3   data/ravdess\Actor_01\03-01-01-01-02-02-01.wav  neutral  ravdess_01   
12  data/ravdess\Actor_01\03-01-03-01-